# 🔍 AIOAC Agent Inspector — AWS / SageMaker Full Visibility Notebook

**Kya karta hai yeh notebook:**
- Har tool ka input + output dikhata hai (live table ke saath)
- AWS IAM permissions check karta hai (S3, SageMaker, IAM self-check)
- Agent ka step-by-step history dikhata hai with gate decisions
- Real SageMaker job chalata hai aur beech mein rokne ka button deta hai
- Cost estimate (cents) dikhata hai har stage pe

---
> ⚠️ **Prerequisites:** `.env` mein `SAGEMAKER_BUCKET`, `SAGEMAKER_ROLE_ARN`, AWS credentials (profile `aioac`) set hone chahiye.

## Cell 1 — Setup & Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)  # override=True ensures that the .env file is reloaded even if it was loaded before

In [ ]:
import sys, os, json, time, threading, textwrap
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# Project root pe path add karo
BACKEND = Path("E:/AIOAC/backend").resolve()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

# .env load karo
from dotenv import load_dotenv
load_dotenv(BACKEND / ".env")

print("✅ Imports OK")
print(f"   Python: {sys.version.split()[0]}")
print(f"   Backend path: {BACKEND}")

## Cell 2 — Settings Preview

In [ ]:
from app.config import settings

SAFE_SETTINGS = {
    "training_backend":      settings.training_backend,
    "sagemaker_region":      settings.sagemaker_region,
    "sagemaker_bucket":      settings.sagemaker_bucket or "❌ NOT SET",
    "sagemaker_role_arn":    settings.sagemaker_role_arn or "❌ NOT SET",
    "sagemaker_instance":    settings.sagemaker_instance_type,
    "sagemaker_sklearn_ver": settings.sagemaker_sklearn_version,
    "sagemaker_poll_s":      settings.sagemaker_poll_seconds,
    "max_job_cap_cents":     settings.max_job_cap_cents,
    "aws_budget_cents":      settings.aws_budget_cents,
    "llm_provider":          settings.llm_provider,
    "agent_max_steps":       settings.agent_max_steps,
}

print("\n📋 Active Settings:")
for k, v in SAFE_SETTINGS.items():
    icon = "✅" if v and "NOT SET" not in str(v) else "❌"
    print(f"   {icon}  {k:30s} = {v}")

## Cell 3 — AWS Credentials & IAM Identity Check

In [ ]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

def check_aws_identity():
    """STS se caller identity lo — credentials valid hain ya nahi."""
    try:
        sts = boto3.client("sts", region_name=settings.sagemaker_region)
        identity = sts.get_caller_identity()
        print("✅ AWS Credentials Valid")
        print(f"   UserId  : {identity['UserId']}")
        print(f"   Account : {identity['Account']}")
        print(f"   ARN     : {identity['Arn']}")
        return identity
    except NoCredentialsError:
        print("❌ No AWS credentials found!")
        print("   Run: aws configure --profile aioac")
        return None
    except ClientError as e:
        print(f"❌ AWS Error: {e}")
        return None

identity = check_aws_identity()

## Cell 4 — AWS Permission Cards (S3 + SageMaker)

In [ ]:
def _check(label, fn):
    """Ek permission check karo aur result print karo."""
    try:
        fn()
        print(f"   ✅  {label}")
        return True
    except ClientError as e:
        code = e.response["Error"]["Code"]
        msg  = e.response["Error"]["Message"]
        print(f"   ❌  {label}  →  {code}: {msg}")
        return False
    except Exception as e:
        print(f"   ⚠️  {label}  →  {type(e).__name__}: {e}")
        return False

BUCKET   = settings.sagemaker_bucket
REGION   = settings.sagemaker_region
ROLE_ARN = settings.sagemaker_role_arn

s3  = boto3.client("s3",         region_name=REGION)
sm  = boto3.client("sagemaker",  region_name=REGION)
iam = boto3.client("iam",        region_name=REGION)

print("\n📦 S3 Permissions:")
_check("s3:ListBucket",          lambda: s3.list_objects_v2(Bucket=BUCKET, MaxKeys=1))
_check("s3:PutObject (test)",    lambda: s3.put_object(Bucket=BUCKET, Key="_permission_check.txt", Body=b"test"))
_check("s3:GetObject (test)",    lambda: s3.get_object(Bucket=BUCKET, Key="_permission_check.txt"))
_check("s3:DeleteObject (test)", lambda: s3.delete_object(Bucket=BUCKET, Key="_permission_check.txt"))

print("\n🤖 SageMaker Permissions:")
_check("sagemaker:ListTrainingJobs",  lambda: sm.list_training_jobs(MaxResults=1))
_check("sagemaker:ListTransformJobs", lambda: sm.list_transform_jobs(MaxResults=1))

print("\n👤 IAM / Role:")
if ROLE_ARN:
    role_name = ROLE_ARN.split("/")[-1]
    _check(f"iam:GetRole ({role_name})", lambda: iam.get_role(RoleName=role_name))
else:
    print("   ⚠️  SAGEMAKER_ROLE_ARN not set — skipping IAM check")

print("\n✅ Permission card done")

## Cell 5 — SageMaker Cost Calculator

In [ ]:
from app.integrations.aws import sagemaker_cost as costs

INSTANCE = settings.sagemaker_instance_type

print(f"💰 Cost Calculator — Instance: {INSTANCE}")
print(f"   Fixed fees  : {costs.fixed_fees_cents()} cents")

for cap in [100, 200, 300, 500, 700]:
    try:
        secs = costs.max_runtime_seconds(cap, INSTANCE)
        cost_if_full = costs.actual_cost_cents(secs, INSTANCE, cap)
        print(f"   Cap {cap:4d}¢  →  max runtime {secs:5d}s  (~{secs//60}min)  |  cost if full = {cost_if_full}¢")
    except costs.CapTooSmall as e:
        print(f"   Cap {cap:4d}¢  →  ❌ CapTooSmall: {e}")

## Cell 6 — Agent Tools Directory (Har Tool ka Card)

In [ ]:
from app.services.agent.tools import TOOLS
from app.services import gate

print(f"🛠  Total tools: {len(TOOLS)}")
print()

LEVEL_LABEL = {0: "🟢 Free (no approval)", 1: "🟡 Needs user approval", 2: "🔴 Requires payment + gate"}

for name, tool in TOOLS.items():
    level = gate.LEVEL[tool.action]
    params = list(tool.parameters.get("properties", {}).keys())
    required = tool.parameters.get("required", [])
    print(f"┌─ {name}")
    print(f"│  Action  : {tool.action.value}")
    print(f"│  Level   : {LEVEL_LABEL[level]}")
    print(f"│  Params  : {params}  (required: {required})")
    print(f"│  Desc    : {textwrap.fill(tool.description, width=70, subsequent_indent='│           ')}")
    print(f"└{'─'*60}")
    print()

## Cell 7 — Instrumented Tool Wrapper (Har tool call ka input/output track karo)

In [ ]:
TOOL_HISTORY = []  # global history — har cell se accessible

def make_instrumented_tools(original_tools: dict) -> dict:
    """Har tool ko wrap karo taaki input/output record ho."""
    from app.services.agent.tools import Tool
    import copy
    wrapped = {}
    for name, tool in original_tools.items():
        original_fn = tool.fn
        def make_wrapper(n, fn):
            def wrapper(ctx, args):
                t0 = time.monotonic()
                ts = datetime.now(timezone.utc).isoformat()
                entry = {"step": len(TOOL_HISTORY) + 1, "tool": n, "args": copy.deepcopy(args),
                         "ts": ts, "result": None, "error": None, "ms": None}
                TOOL_HISTORY.append(entry)
                try:
                    result = fn(ctx, args)
                    entry["result"] = result
                    entry["ms"] = int((time.monotonic() - t0) * 1000)
                    return result
                except Exception as exc:
                    entry["error"] = f"{type(exc).__name__}: {exc}"
                    entry["ms"] = int((time.monotonic() - t0) * 1000)
                    raise
            return wrapper
        wrapped[name] = Tool(
            name=tool.name, description=tool.description,
            parameters=tool.parameters, action=tool.action,
            fn=make_wrapper(name, original_fn)
        )
    return wrapped

def print_tool_history():
    """Recorded history print karo."""
    if not TOOL_HISTORY:
        print("(No tool calls recorded yet)")
        return
    for e in TOOL_HISTORY:
        status = "❌ ERROR" if e["error"] else "✅ OK"
        print(f"\n[Step {e['step']}] {e['tool']}  {status}  ({e['ms']}ms)  @ {e['ts'][11:19]}")
        print(f"  INPUT  : {json.dumps(e['args'], default=str)[:300]}")
        if e["error"]:
            print(f"  ERROR  : {e['error'][:300]}")
        else:
            out = json.dumps(e["result"], default=str)
            print(f"  OUTPUT : {out[:400]}{'...' if len(out) > 400 else ''}")

print("✅ Instrumented tool wrapper ready")

## Cell 8 — SageMaker Backend with Live Stop Button

Yahan real SageMaker job chalega. **STOP** button beech mein dabao job rukne ke liye.

In [ ]:
import threading
from app.integrations.aws.sagemaker import SageMakerBackend, SageMakerJobError

# ── UI widgets ──────────────────────────────────────────────────────────
stop_btn     = widgets.Button(description="⛔ STOP Training", button_style="danger", layout=widgets.Layout(width="200px"))
status_label = widgets.Label(value="Status: idle")
progress_bar = widgets.FloatProgress(value=0, min=0, max=1, description="Progress:", layout=widgets.Layout(width="500px"))
log_output   = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", max_height="400px", overflow_y="auto"))

display(widgets.HBox([stop_btn, status_label]))
display(progress_bar)
display(log_output)

# ── Stop flag ────────────────────────────────────────────────────────────
_should_stop = threading.Event()

def on_stop_clicked(b):
    _should_stop.set()
    status_label.value = "Status: ⛔ stop requested — waiting for AWS..."
    stop_btn.disabled = True

stop_btn.on_click(on_stop_clicked)

# ── Progress callback ────────────────────────────────────────────────────
STAGE_FRACTION = {"queued": 0.05, "provisioning": 0.15, "preparing data": 0.30,
                  "training": 0.70, "saving the model": 0.95, "stopping": 0.98}

def on_progress(stage: str, fraction: float):
    progress_bar.value = fraction
    status_label.value = f"Status: {stage}  ({fraction*100:.0f}%)"
    with log_output:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {stage:25s}  {fraction*100:5.1f}%")

print("✅ SageMaker UI widgets ready. Run the next cell to start a job.")

## Cell 9 — 🚀 Real SageMaker Job Chalao

Agar dataset file nahi hai to titanic sample automatically use hoga.
**Stop button anytime daba sakte ho.**

In [ ]:
import tempfile
import pandas as pd
import numpy as np

# ── Synthetic dataset — replace with your real CSV path ─────────────────
DATASET_PATH = None   # e.g. "E:/AIOAC/backend/storage/my_dataset.csv"
TARGET_COL   = "survived"
CAP_CENTS    = 300   # max spend (3 dollars)

if DATASET_PATH is None:
    # Tiny synthetic dataset (titanic-style) as a fallback
    rng = np.random.default_rng(42)
    n = 800
    df = pd.DataFrame({
        "pclass":   rng.choice([1, 2, 3], n),
        "age":      rng.uniform(1, 80, n).round(1),
        "fare":     rng.uniform(5, 500, n).round(2),
        "sex":      rng.choice(["male", "female"], n),
        "sibsp":    rng.integers(0, 6, n),
        "survived": rng.integers(0, 2, n),
    })
    tmp = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
    df.to_csv(tmp.name, index=False)
    DATASET_PATH = tmp.name
    print(f"📊 Using synthetic dataset: {n} rows, target='{TARGET_COL}'")
    print(f"   Temp file: {DATASET_PATH}")
else:
    print(f"📊 Using dataset: {DATASET_PATH}")

# ── Reset stop flag & UI ─────────────────────────────────────────────────
_should_stop.clear()
stop_btn.disabled = False
progress_bar.value = 0
status_label.value = "Status: starting..."
log_output.clear_output()
TOOL_HISTORY.clear()

# ── Build backend ────────────────────────────────────────────────────────
backend = SageMakerBackend()

print(f"\n🚀 Starting SageMaker Job")
print(f"   Instance : {backend.instance_type}")
print(f"   Bucket   : {backend.bucket}")
print(f"   Role     : {backend.role_arn}")
print(f"   Cap      : {CAP_CENTS} cents")

t_start = time.time()
result  = None
error   = None

try:
    result = backend.run(
        dataset_path = DATASET_PATH,
        target       = TARGET_COL,
        cap_cents    = CAP_CENTS,
        on_progress  = on_progress,
        should_stop  = lambda: _should_stop.is_set(),
        job_key      = f"inspector-{int(time.time())}",
    )
except SageMakerJobError as e:
    error = e
    status_label.value = f"Status: ❌ {e.kind}: {e}"

elapsed = time.time() - t_start

print(f"\n{'='*60}")
if result:
    print(f"✅ Job finished in {elapsed:.0f}s")
    print(f"   accuracy              : {result.accuracy:.4f}")
    print(f"   simple_guess_accuracy : {result.simple_guess_accuracy:.4f}")
    print(f"   runtime_seconds       : {result.runtime_seconds:.1f}")
    print(f"   cost_cents            : {result.cost_cents}¢")
    print(f"   stopped_at_cap        : {result.stopped_at_cap}")
    print(f"   user_stopped          : {result.stopped}")
    print(f"   model_bytes           : {len(result.model_bytes) if result.model_bytes else 'None'} bytes")
    status_label.value = f"Status: ✅ Done  accuracy={result.accuracy:.3f}  cost={result.cost_cents}¢"
    progress_bar.value = 1.0
elif error:
    print(f"❌ Job failed: kind={error.kind}  cost_cents={error.cost_cents}")
    print(f"   {error}")
else:
    print("⚠️  Job ended with no result and no error (unexpected)")

if backend.last_warnings:
    print("\n⚠️  Warnings:")
    for w in backend.last_warnings:
        print(f"   {w}")

## Cell 10 — Agent History Viewer (Live RunLog replay)

In [ ]:
from app.services.agent.runlog import RunLog, MarkdownSink, ConsoleSink, format_line

# In-memory log — agent ke events collect karta hai
HISTORY_EVENTS = []

class InspectorSink:
    """Har event ko HISTORY_EVENTS mein store karta hai."""
    def write(self, event: dict) -> None:
        HISTORY_EVENTS.append(event)

inspector_log = RunLog(sinks=[InspectorSink(), ConsoleSink(min_level="DEBUG")])

# Dummy events simulate karo (real agent run pe use hoga — see Cell 11)
inspector_log.begin_turn("project-test", "run-001")
inspector_log.emit("turn_start", status_before="new", message_chars=42, approved=[], model="gpt-test")
inspector_log.emit("tool_call",  call_id="c1",  name="search_datasets", args={"query": "churn prediction"})
inspector_log.emit("gate",       action="search", level_num=0, allowed=True, reason=None)
inspector_log.emit("tool_result", call_id="c1", name="search_datasets", ms=120, ok=True,
                   summary="3 results", result={"results": [{"id": "hf:aioac/sample-churn"}]})
inspector_log.emit("turn_end",   status="reply", reason=None, steps_used=1, tokens_used=1200, cost_usd=0.002)

print(f"\n📜 Agent History ({len(HISTORY_EVENTS)} events)")
print(f"{'─'*70}")
for ev in HISTORY_EVENTS:
    print(format_line(ev))

## Cell 11 — Full Agent Run with Inspector

Poora agent chalao — database, LLM, tools sab real.
Isko chalane ke liye ek real project_id chahiye (DB se).

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
PROJECT_ID   = None   # "<paste a real project_id from the DB>"  — None = skip
USER_MESSAGE = "Find me a churn prediction dataset and tell me what columns it has."

if PROJECT_ID is None:
    print("⏭  Skipping full agent run — set PROJECT_ID to a real project id from the DB.")
    print("   Example: PROJECT_ID = '01J...'")
else:
    from app.db.session import get_engine
    from sqlalchemy.orm import Session as SASession
    from app.integrations.factory import make_tool_context
    from app.integrations.llm.azure import AzureLLMClient
    from app.services.agent.loop import run_agent
    from app.services.agent.runlog import RunLog, ConsoleSink
    from app.services.agent.tools import TOOLS

    HISTORY_EVENTS.clear()
    TOOL_HISTORY.clear()

    # Instrumented tools inject karo
    import app.services.agent.tools as _tools_module
    _original_tools = dict(_tools_module.TOOLS)
    _tools_module.TOOLS.update(make_instrumented_tools(_original_tools))

    log = RunLog(sinks=[InspectorSink(), ConsoleSink(min_level="INFO")])

    engine = get_engine()
    with SASession(engine) as session:
        ctx = make_tool_context(session, PROJECT_ID)
        llm = AzureLLMClient()
        result_ag = run_agent(ctx, llm, USER_MESSAGE, log=log)

    # Restore original tools
    _tools_module.TOOLS.update(_original_tools)

    print(f"\n🏁 Agent result: status={result_ag.status}")
    if result_ag.text:
        print(f"   Reply: {result_ag.text[:500]}")
    if result_ag.card:
        print(f"   Card: {result_ag.card}")

    print(f"\n📜 Agent History ({len(HISTORY_EVENTS)} events):")
    print("─" * 70)
    for ev in HISTORY_EVENTS:
        print(format_line(ev))

    print(f"\n🛠  Tool Call History ({len(TOOL_HISTORY)} calls):")
    print_tool_history()

## Cell 12 — Tool History Pretty Viewer

In [ ]:
print_tool_history()

## Cell 13 — Live SageMaker Job Monitor (Job naam se)

In [ ]:
JOB_TO_MONITOR = None  # "aioac-20261028-012345-ab1234"  — paste a real job name

if JOB_TO_MONITOR is None:
    print("⏭  Set JOB_TO_MONITOR = '<job-name>' to watch a running job.")
else:
    sm_client = boto3.client("sagemaker", region_name=settings.sagemaker_region)
    stop_monitor_btn = widgets.Button(description="⛔ STOP this job", button_style="danger")
    monitor_out = widgets.Output()
    display(stop_monitor_btn, monitor_out)

    def stop_job(b):
        sm_client.stop_training_job(TrainingJobName=JOB_TO_MONITOR)
        with monitor_out:
            print(f"⛔ Stop sent to {JOB_TO_MONITOR}")
        stop_monitor_btn.disabled = True

    stop_monitor_btn.on_click(stop_job)

    WATCH_FIELDS = ["TrainingJobStatus", "SecondaryStatus", "TrainingTimeInSeconds",
                    "BillableTimeInSeconds", "FailureReason"]

    with monitor_out:
        while True:
            desc = sm_client.describe_training_job(TrainingJobName=JOB_TO_MONITOR)
            status = desc["TrainingJobStatus"]
            clear_output(wait=True)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Job: {JOB_TO_MONITOR}")
            for f in WATCH_FIELDS:
                if f in desc and desc[f] is not None:
                    print(f"   {f:35s}: {desc[f]}")
            if status in ("Completed", "Failed", "Stopped"):
                print(f"\n🏁 Final status: {status}")
                cost = costs.actual_cost_cents(
                    desc.get("BillableTimeInSeconds") or 0,
                    settings.sagemaker_instance_type,
                    settings.max_job_cap_cents
                )
                print(f"   Estimated cost : {cost} cents")
                break
            time.sleep(settings.sagemaker_poll_seconds)

## Cell 14 — Recent SageMaker Jobs (Last 10)

In [ ]:
sm_client = boto3.client("sagemaker", region_name=settings.sagemaker_region)

try:
    resp = sm_client.list_training_jobs(MaxResults=10, SortBy="CreationTime", SortOrder="Descending")
    jobs = resp.get("TrainingJobSummaries", [])
    print(f"📋 Last {len(jobs)} SageMaker training jobs in {settings.sagemaker_region}:")
    print(f"{'Name':45s} {'Status':12s} {'Created':25s} {'Billable(s)'}")
    print("-" * 100)
    for j in jobs:
        name    = j["TrainingJobName"]
        status  = j["TrainingJobStatus"]
        created = j["CreationTime"].strftime("%Y-%m-%d %H:%M:%S")
        billable = j.get("TrainingEndTime", "")  # shown as end time; billable only from describe
        print(f"{name:45s} {status:12s} {created:25s}")
except ClientError as e:
    print(f"❌ Could not list jobs: {e}")

## Cell 15 — Summary Dashboard

In [ ]:
print("\n" + "═" * 65)
print("  AIOAC Inspector — Summary")
print("═" * 65)

# Settings
print(f"\n  Backend     : {settings.training_backend}")
print(f"  Instance    : {settings.sagemaker_instance_type}")
print(f"  Region      : {settings.sagemaker_region}")
print(f"  Bucket      : {settings.sagemaker_bucket or '❌ not set'}")

# Tool count
print(f"\n  Tools available : {len(TOOLS)}")
print(f"  Tool calls run  : {len(TOOL_HISTORY)}")
print(f"  Agent events    : {len(HISTORY_EVENTS)}")

# Last training result
if result:
    print(f"\n  Last job result :")
    print(f"    accuracy = {result.accuracy:.4f}")
    print(f"    cost     = {result.cost_cents} cents")
    print(f"    runtime  = {result.runtime_seconds:.1f}s")
    print(f"    stopped  = {result.stopped}")
else:
    print("\n  Last job result : (no job run this session)")

# Tool history summary
if TOOL_HISTORY:
    print(f"\n  Tool call log:")
    for e in TOOL_HISTORY:
        ok = "✅" if not e["error"] else "❌"
        print(f"    {ok} [{e['step']}] {e['tool']:25s}  {e['ms']}ms")

print("\n" + "═" * 65)